# Fine-tune Qwen2.5-3B for OLX Car Description Extraction

Training data: 1057 Portuguese car listing descriptions → 24-field JSON.

**Fields extracted:** desc_mentions_num_owners, desc_mentions_accident, accident_details,
service_history, desc_mentions_repair, repair_details, mileage_in_description_km,
desc_mentions_customs_cleared, imported, right_hand_drive, mechanical_condition, paint_condition,
suspicious_signs, extras, issues, reason_for_sale, **urgency, warranty, tuning_or_mods,
taxi_fleet_rental, recent_maintenance, tires_condition, first_owner_selling**

**Steps:**
1. Upload `training_data.jsonl`
2. Fine-tune with Unsloth + QLoRA (Qwen2.5-3B-Instruct)
3. Export to GGUF Q4_K_M for Ollama
4. Download and deploy

In [ ]:
# 1. Install Unsloth (optimized for free Colab T4)
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 1b. Verify GPU
import torch
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu}, VRAM: {vram:.1f} GB")
if vram < 14:
    print("WARNING: <14GB VRAM — may need batch_size=1 in training step")

In [ ]:
# 2. Upload training data
from google.colab import files
uploaded = files.upload()  # upload training_data.jsonl

In [ ]:
# 3. Load and prepare dataset
import json
from datasets import Dataset

data = []
with open("training_data.jsonl") as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} training examples")

# Verify new fields are present
sample_output = json.loads(data[0]["messages"][1]["content"])
expected_new = ["urgency", "warranty", "tuning_or_mods", "taxi_fleet_rental",
                "recent_maintenance", "tires_condition", "first_owner_selling"]
missing = [f for f in expected_new if f not in sample_output]
if missing:
    print(f"WARNING: missing new fields: {missing}")
else:
    print(f"All 24 fields present ✓")
    print(f"Sample new fields: urgency={sample_output['urgency']}, "
          f"tuning={sample_output['tuning_or_mods']}, "
          f"maintenance={sample_output['recent_maintenance']}")

In [ ]:
# 4. Load model with Unsloth (4-bit quantized)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

print(f"Model loaded: Qwen2.5-3B-Instruct")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# 5. Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                     # LoRA rank (higher for 3B — more capacity)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,           # optimized, no dropout
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# 6. Format dataset for chat template
def format_chat(example):
    """Apply Qwen2.5 chat template to training example."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_chat)

# Check token lengths
lengths = [len(tokenizer.encode(x["text"])) for x in dataset]
print(f"Token lengths — min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)/len(lengths):.0f}")
print(f"\nFormatted sample (first 300 chars):\n{dataset[0]['text'][:300]}")

In [ ]:
# 7. Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,    # 3B needs more VRAM, reduce batch
        gradient_accumulation_steps=8,    # effective batch = 16 (same as before)
        warmup_steps=20,
        num_train_epochs=3,
        learning_rate=1.5e-4,             # slightly lower for 3B stability
        fp16=True,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        save_strategy="epoch",
    ),
)

print(f"Training {len(dataset)} examples, 3 epochs, effective batch=16...")
stats = trainer.train()
print(f"\nDone! Final loss: {stats.training_loss:.4f}")

In [ ]:
# 8. Quick test before export
FastLanguageModel.for_inference(model)

EXTRACTION_PROMPT = """\
Extract structured data from this Portuguese car listing. JSON fields (null if unknown):
desc_mentions_num_owners(int), desc_mentions_accident(bool), accident_details(str), service_history(bool), \
desc_mentions_repair(bool), repair_details(str), \
mileage_in_description_km(int), desc_mentions_customs_cleared(bool), imported(bool), \
right_hand_drive(bool), \
mechanical_condition("excellent"/"good"/"fair"/"poor"), paint_condition(same), \
suspicious_signs(list), extras(list), issues(list), reason_for_sale(str), \
urgency("high"/"medium"/"low"), warranty(bool), tuning_or_mods(list), \
taxi_fleet_rental(bool), recent_maintenance(list), tires_condition("new"/"good"/"fair"/"poor"), \
first_owner_selling(bool).
Rules: mileage_in_description_km=exact km as integer ("4300 km"→4300, "150 mil km"→150000, \
"89.500km"→89500, "4.300km"→4300). "mil"=thousand ONLY when written as a separate word. \
desc_mentions_repair=true if ANY repair/damage mentioned. desc_mentions_accident=true if collision mentioned. \
desc_mentions_customs_cleared=look for "desalfandegado","legalizado","por legalizar". \
right_hand_drive=true if right-hand drive/UK/Japan import/"mão inglesa"/"volante à direita"/"condução à direita". \
urgency=high if "urgente","preciso vender rápido","emigração","preço para despachar"; medium if "aceito propostas","negociável","oportunidade"; low otherwise. \
warranty=true if "garantia" mentioned (not "sem garantia"). \
tuning_or_mods=list of aftermarket modifications: "reprogramação","stage 1/2","remap","chip tuning","suspensão rebaixada","coilovers","escape desportivo","downpipe","wrap","bodykit". \
taxi_fleet_rental=true if "ex-táxi","TVDE","Uber","Bolt","rent-a-car","frota","carro de empresa". \
recent_maintenance=list of specific completed maintenance: "correia distribuição","embreagem nova","travões novos","revisão feita", with km/date if mentioned. \
tires_condition="new" if "pneus novos"; "good" if "bom estado"/"80% piso"; "fair" if "razoável"/"50%"; "poor" if "gastos"/"precisa pneus". \
first_owner_selling=true if "1 dono desde novo","único dono","comprado novo por mim","vendo o meu".

"""

test_desc = """BMW 320d Pack M 2015 — Único dono desde novo.
Revisões sempre na marca. Correia de distribuição trocada aos 150.000km.
Pneus novos Michelin. Sem acidentes.
Reprogramação stage 1, 210cv. Escape desportivo Akrapovic.
Aceito propostas razoáveis. Garantia de motor 6 meses."""

messages = [{"role": "user", "content": EXTRACTION_PROMPT + test_desc}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)
response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("Test output:")
try:
    parsed = json.loads(response)
    for k, v in parsed.items():
        print(f"  {k}: {v}")
    # Validate new fields
    print("\n--- New fields check ---")
    assert parsed.get("urgency") == "medium", f"Expected medium, got {parsed.get('urgency')}"
    assert parsed.get("warranty") == True, f"Expected True, got {parsed.get('warranty')}"
    assert "stage 1" in str(parsed.get("tuning_or_mods", [])).lower(), "Missing stage 1 in tuning"
    assert parsed.get("tires_condition") == "new", f"Expected new, got {parsed.get('tires_condition')}"
    assert parsed.get("first_owner_selling") == True, f"Expected True, got {parsed.get('first_owner_selling')}"
    print("All new fields correct ✓")
except json.JSONDecodeError:
    print("WARNING: Output is not valid JSON!")
    print(response)

In [ ]:
# 9. Export to GGUF for Ollama
# Q4_K_M — best balance of speed and quality for M1 8GB
model.save_pretrained_gguf(
    "car-parser-3b",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported: car-parser-3b/")

In [ ]:
# 10. Create Ollama Modelfile
modelfile = """FROM ./car-parser-3b-Q4_K_M.gguf

PARAMETER temperature 0.1
PARAMETER num_predict 512
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
"""

with open("car-parser-3b/Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile created!")
print("\n--- Deploy instructions ---")
print("1. Download car-parser-3b/ folder")
print("2. cd car-parser-3b/")
print("3. ollama create car-parser-3b -f Modelfile")
print("4. ollama run car-parser-3b")
print("\nThen update config/settings.yaml:")
print('  ollama_model: "car-parser-3b"')

In [ ]:
# 11. Download GGUF + Modelfile
import shutil
shutil.make_archive("car-parser-3b", "zip", "car-parser-3b")
files.download("car-parser-3b.zip")